<a href="https://colab.research.google.com/github/dannm2008/gemelo-digital-glucosa/blob/main/Gemelo_Digital_D1NAMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub -q
import kagglehub

In [ ]:
!rm -rf diabetes_subset_pictures* datos_d1namo
print("¡Limpieza completada! El archivo corrupto fue eliminado.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Importar algoritmos para la comparativa
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
try:
    from xgboost import XGBRegressor
    xgboost_disponible = True
except ImportError:
    xgboost_disponible = False

# Importar herramientas para la interfaz interactiva
import ipywidgets as widgets
from IPython.display import display, clear_output

# =====================================================================
#  LÍNEA MÁGICA: Asegurar la descarga automática si Colab borró el archivo
# =====================================================================
ruta_final = 'datos_d1namo/glucose.csv'

if not os.path.exists(ruta_final):
    print(" El archivo no se encontró en la sesión actual de Colab.")
    print(" Descargando y reestructurando el dataset automáticamente con kagglehub...")
    import kagglehub
    import shutil

    # Descargar desde la fuente original de Kaggle
    ruta_cache = kagglehub.dataset_download("sarabhian/d1namo-ecg-glucose-data")

    # Ruta específica del Paciente 005
    ruta_paciente_005 = os.path.join(ruta_cache, 'healthy_subset_pictures-glucose-food', 'healthy_subset_pictures-glucose-food', '005')
    archivo_origen = os.path.join(ruta_paciente_005, 'glucose.csv')

    # Recrear la carpeta local para que tu proyecto no se rompa
    os.makedirs('datos_d1namo', exist_ok=True)
    shutil.copy(archivo_origen, ruta_final)
    print(" ¡Dataset recuperado con éxito! Continuando con el entrenamiento...\n")

# =====================================================================
# 1. Cargar y procesar datos del Paciente 005 (Dataset D1NAMO)
# =====================================================================
df = pd.read_csv(ruta_final)
df['glucose_mgdl'] = df['glucose'] * 18.016

# Preparar variables contextuales de tiempo (minutos del día) y comida (type)
df['minutos_dia'] = pd.to_datetime(df['time'], format='%H:%M').dt.hour * 60 + pd.to_datetime(df['time'], format='%H:%M').dt.minute
df_ia = pd.get_dummies(df, columns=['type'], drop_first=False)

# Definir columnas de entrada (X) y objetivo (y)
columnas_predictoras = ['minutos_dia'] + [col for col in df_ia.columns if 'type_' in col]
X = df_ia[columnas_predictoras]
y = df_ia['glucose_mgdl']

# División de datos (80% entrenamiento, 20% evaluación)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("==================================================")
print("  FASE: ENTRENANDO Y EVALUANDO MODELOS DE IA ")
print("==================================================")

# --- MODELO 1: Árbol de Decisión ---
modelo_dt = DecisionTreeRegressor(random_state=42)
modelo_dt.fit(X_train, y_train)
pred_dt = modelo_dt.predict(X_test)
mae_dt = mean_absolute_error(y_test, pred_dt)
print(f"• MAE Árbol de Decisión: {mae_dt:.2f} mg/dL")

# --- MODELO 2: Random Forest (El modelo principal para el TIR) ---
modelo_rf = RandomForestRegressor(n_estimators=50, random_state=42)
modelo_rf.fit(X_train, y_train)
pred_rf = modelo_rf.predict(X_test)
mae_rf = mean_absolute_error(y_test, pred_rf)
print(f"• MAE Random Forest:     {mae_rf:.2f} mg/dL")

# --- MODELO 3: XGBoost ---
if xgboost_disponible:
    modelo_xgb = XGBRegressor(n_estimators=50, random_state=42)
    modelo_xgb.fit(X_train, y_train)
    pred_xgb = modelo_xgb.predict(X_test)
    mae_xgb = mean_absolute_error(y_test, pred_xgb)
    print(f"• MAE XGBoost Regressor: {mae_xgb:.2f} mg/dL")
else:
    mae_xgb = None
    print("• XGBoost no está disponible en este entorno de Colab (Se omitirá en la gráfica)")
print("==================================================")

# 2. Desplegar Gráfica Comparativa (Responde a la Semana 6)
plt.figure(figsize=(11, 4))
plt.plot(y_test.values, label='Valores Reales', color='black', marker='o', linewidth=2.5)
plt.plot(pred_dt, label=f'Árbol de Decisión (MAE: {mae_dt:.2f})', linestyle=':', marker='s', color='crimson')
plt.plot(pred_rf, label=f'Random Forest (MAE: {mae_rf:.2f})', linestyle='--', marker='x', color='dodgerblue')
if xgboost_disponible:
    plt.plot(pred_xgb, label=f'XGBoost (MAE: {mae_xgb:.2f})', linestyle='-.', marker='^', color='darkorange')

plt.title('Comparativa de Modelos de IA (Paciente 005 - Gemelo Digital)', fontsize=12, fontweight='bold')
plt.xlabel('Muestras de Evaluación', fontsize=10)
plt.ylabel('Glucosa (mg/dL)', fontsize=10)
plt.ylim(50, 150)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()

# 3. INTERFAZ INTERACTIVA DE PREDICCIÓN
print("\n==================================================")
print("   INTERFAZ DE MONITOREO DEL GEMELO DIGITAL")
print("==================================================\n")

# Extraer los tipos de comida reales que conoce el modelo
lista_comidas = [col.replace('type_', '') for col in columnas_predictoras if 'type_' in col]

# Crear componentes visuales (Controles para la interfaz)
selector_hora = widgets.IntSlider(value=480, min=0, max=1439, step=15, description='Hora (Minutos):', style={'description_width': 'initial'})
selector_comida = widgets.Dropdown(options=lista_comidas, value=lista_comidas[0], description='Momento/Comida:', style={'description_width': 'initial'})
boton_predecir = widgets.Button(description='Calcular Predicción', button_style='success', icon='calculator')
salida_resultado = widgets.Output()

# Función que ejecuta la predicción al presionar el botón
def realizar_prediccion(b):
    with salida_resultado:
        clear_output()

        # Traducir los minutos a formato de hora amigable (HH:MM)
        horas = selector_hora.value // 60
        minutos = selector_hora.value % 60
        hora_formateada = f"{horas:02d}:{minutos:02d}"

        # Construir el vector de entrada con la misma estructura que X
        datos_entrada = pd.DataFrame(0, index=[0], columns=columnas_predictoras)
        datos_entrada['minutos_dia'] = selector_hora.value

        columna_comida_activa = f"type_{selector_comida.value}"
        if columna_comida_activa in datos_entrada.columns:
            datos_entrada[columna_comida_activa] = 1

        # Ejecutar la predicción con el modelo de Random Forest (el mejor evaluado)
        resultado_glucosa = modelo_rf.predict(datos_entrada)[0]

        # Mostrar los resultados estilizados en pantalla
        print(f" Escenario Evaluado: {hora_formateada} hrs |  Estado: {selector_comida.value}")
        print(f" Predicción de Glucosa Estimada por IA: {resultado_glucosa:.2f} mg/dL")

        # Alerta clínica básica según el resultado
        if resultado_glucosa < 70:
            print(" Alerta Clínica: Riesgo de Hipoglucemia detectado.")
        elif resultado_glucosa > 140:
            print(" Alerta Clínica: Riesgo de Hiperglucemia postprandial detectado.")
        else:
            print(" Estado Metabólico: Valores en rango seguro (70-140 mg/dL).")

# Conectar el botón con la función de predicción
boton_predecir.on_click(realizar_prediccion)

# Desplegar la interfaz en la pantalla de Colab
layout_controles = widgets.VBox([selector_hora, selector_comida, boton_predecir, salida_resultado])
display(layout_controles)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor, plot_tree  # plot_tree dibuja las reglas de decisión en pantalla
import kagglehub

print(" Iniciando Fase de Explicabilidad (Semana 5)...")
print(" Cargando datos y entrenando el ÁRBOL DE DECISIÓN POBLACIONAL...")

# 1. Localizar el dataset completo de forma segura en el caché rápido de kagglehub
ruta_cache = kagglehub.dataset_download("sarabhian/d1namo-ecg-glucose-data")

# Función optimizada para cargar los CSV desde el caché sin saturar el almacenamiento local
def cargar_csv_paciente(id_paciente):
    ruta_csv = os.path.join(ruta_cache, 'healthy_subset_pictures-glucose-food', 'healthy_subset_pictures-glucose-food', id_paciente, 'glucose.csv')
    if os.path.exists(ruta_csv):
        df_p = pd.read_csv(ruta_csv)
        df_p['glucose_mgdl'] = df_p['glucose'] * 18.016
        df_p['minutos_dia'] = pd.to_datetime(df_p['time'], format='%H:%M').dt.hour * 60 + pd.to_datetime(df_p['time'], format='%H:%M').dt.minute

        # Estandarizar columnas de comidas (One-Hot Encoding)
        for comida in ['AB', 'AD', 'AL', 'BB', 'BD', 'BL', 'M']:
            df_p[f'type_{comida}'] = (df_p['type'] == comida).astype(int)

        columnas_X = ['minutos_dia'] + [f'type_{c}' for c in ['AB', 'AD', 'AL', 'BB', 'BD', 'BL', 'M']]
        return df_p[columnas_X], df_p['glucose_mgdl']
    return None

# =====================================================================
# 2. COMBINAR DATOS DE LA POBLACIÓN BASE (Pacientes 001, 002, 003)
# =====================================================================
X_pob_list, y_pob_list = [], []
for p_id in ['001', '002', '003']:
    datos = cargar_csv_paciente(p_id)
    if datos is not None:
        X_pob_list.append(datos[0])
        y_pob_list.append(datos[1])

X_poblacional = pd.concat(X_pob_list, ignore_index=True)
y_poblacional = pd.concat(y_pob_list, ignore_index=True)

# 3. Entrenar el Árbol de Decisión Poblacional Base
# max_depth=3 asegura que el gráfico sea legible y no un manchón negro ilegible
modelo_poblacional_dt = DecisionTreeRegressor(max_depth=3, random_state=42)
modelo_poblacional_dt.fit(X_poblacional, y_poblacional)
print(" Árbol de Decisión Poblacional listo y entrenado.")

# =====================================================================
# 4. EVALUAR EL MODELO POBLACIONAL EN EL PACIENTE TARGET (005)
# =====================================================================
X_005, y_005 = cargar_csv_paciente('005')
pred_pob = modelo_poblacional_dt.predict(X_005)
mae_pob = mean_absolute_error(y_005, pred_pob)

print("\n==================================================")
print("   DESEMPEÑO DEL MODELO POBLACIONAL POBLACIONAL ")
print("==================================================")
print(f"• MAE de la Base Poblacional en Paciente 005: {mae_pob:.2f} mg/dL")
print("==================================================\n")

# =====================================================================
# 5.  GRAFICAR EL ÁRBOL DE DECISIÓN CLÍNICA (EXPLICABILIDAD)
# =====================================================================
plt.figure(figsize=(20, 10), dpi=300) # Alta resolución para leer con claridad absoluta

plot_tree(
    modelo_poblacional_dt,
    feature_names=list(X_poblacional.columns),
    filled=True,             # Colorea las cajas según la glucosa estimada
    rounded=True,            # Bordes redondeados estéticos
    fontsize=10,             # Tamaño de letra óptimo para leer los nodos
    precision=2              # Redondear valores a 2 decimales
)

plt.title('Árbol de Decisión Clínica del Modelo Poblacional (Explicabilidad Algorítmica)', fontsize=16, fontweight='bold', pad=20)
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor, plot_tree
import kagglehub

print(" Iniciando Fase de Personalización (Semana 5)...")
print(" Cargando datos y entrenando el ÁRBOL DE DECISIÓN PERSONALIZADO...")

# 1. Localizar el dataset en el caché rápido
ruta_cache = kagglehub.dataset_download("sarabhian/d1namo-ecg-glucose-data")

def cargar_csv_paciente(id_paciente):
    ruta_csv = os.path.join(ruta_cache, 'healthy_subset_pictures-glucose-food', 'healthy_subset_pictures-glucose-food', id_paciente, 'glucose.csv')
    if os.path.exists(ruta_csv):
        df_p = pd.read_csv(ruta_csv)
        df_p['glucose_mgdl'] = df_p['glucose'] * 18.016
        df_p['minutos_dia'] = pd.to_datetime(df_p['time'], format='%H:%M').dt.hour * 60 + pd.to_datetime(df_p['time'], format='%H:%M').dt.minute

        for comida in ['AB', 'AD', 'AL', 'BB', 'BD', 'BL', 'M']:
            df_p[f'type_{comida}'] = (df_p['type'] == comida).astype(int)

        columnas_X = ['minutos_dia'] + [f'type_{c}' for c in ['AB', 'AD', 'AL', 'BB', 'BD', 'BL', 'M']]
        return df_p[columnas_X], df_p['glucose_mgdl']
    return None

# =====================================================================
# 2. CARGAR Y DIVIDIR DATOS EXCLUSIVOS DEL PACIENTE 005
# =====================================================================
X_005, y_005 = cargar_csv_paciente('005')

# Dividimos en 80% entrenamiento y 20% prueba para evaluar de forma justa
X_train_005, X_test_005, y_train_005, y_test_005 = train_test_split(X_005, y_005, test_size=0.2, random_state=42)

# 3. Entrenar el Árbol de Decisión Personalizado (Misma profundidad max_depth=3)
modelo_personalizado_dt = DecisionTreeRegressor(max_depth=3, random_state=42)
modelo_personalizado_dt.fit(X_train_005, y_train_005)
print(" Árbol de Decisión Personalizado entrenado con éxito.")

# =====================================================================
# 4. EVALUAR EL MODELO PERSONALIZADO
# =====================================================================
pred_pers = modelo_personalizado_dt.predict(X_test_005)
mae_pers = mean_absolute_error(y_test_005, pred_pers)

print("\n==================================================")
print("   DESEMPEÑO DEL MODELO PERSONALIZADO (ÁRBOL) ")
print("==================================================")
print(f"• MAE Personalizado en Paciente 005: {mae_pers:.2f} mg/dL")
print("==================================================\n")

# =====================================================================
# 5.  GRAFICAR EL ÁRBOL DE DECISIÓN PERSONALIZADO (EXPLICABILIDAD)
# =====================================================================
plt.figure(figsize=(20, 10), dpi=300)

plot_tree(
    modelo_personalizado_dt,
    feature_names=list(X_005.columns),
    filled=True,
    rounded=True,
    fontsize=10,
    precision=2
)

plt.title('Árbol de Decisión Clínica Personalizado - Paciente 005 (Estrategia Individual)', fontsize=16, fontweight='bold', pad=20)
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import kagglehub

print(" Configurando entorno y preparando datos para la RED NEURONAL LSTM...")

# 1. Localizar el dataset en el caché rápido
ruta_cache = kagglehub.dataset_download("sarabhian/d1namo-ecg-glucose-data")

def cargar_csv_paciente(id_paciente):
    ruta_csv = os.path.join(ruta_cache, 'healthy_subset_pictures-glucose-food', 'healthy_subset_pictures-glucose-food', id_paciente, 'glucose.csv')
    if os.path.exists(ruta_csv):
        df_p = pd.read_csv(ruta_csv)
        df_p['glucose_mgdl'] = df_p['glucose'] * 18.016
        df_p['minutos_dia'] = pd.to_datetime(df_p['time'], format='%H:%M').dt.hour * 60 + pd.to_datetime(df_p['time'], format='%H:%M').dt.minute

        for comida in ['AB', 'AD', 'AL', 'BB', 'BD', 'BL', 'M']:
            df_p[f'type_{comida}'] = (df_p['type'] == comida).astype(int)

        columnas_X = ['minutos_dia'] + [f'type_{c}' for c in ['AB', 'AD', 'AL', 'BB', 'BD', 'BL', 'M']]
        return df_p[columnas_X].values, df_p['glucose_mgdl'].values
    return None

# =====================================================================
# 2. CARGAR Y NORMALIZAR DATOS (Paciente 005)
# =====================================================================
X_raw, y_raw = cargar_csv_paciente('005')

# Las redes neuronales son muy sensibles a las escalas, así que normalizamos las características
escalador_X = StandardScaler()
X_scaled = escalador_X.fit_transform(X_raw)

# Remodelar los datos a 3D para la LSTM: (muestras, pasos_de_tiempo, caracteristicas)
# Usaremos un paso de tiempo de 1 (evaluación registro por registro en serie)
X_3D = np.reshape(X_scaled, (X_scaled.shape[0], 1, X_scaled.shape[1]))

# División de datos (80% entrenamiento, 20% evaluación)
limite = int(len(X_3D) * 0.8)
X_train_lstm, X_test_lstm = X_3D[:limite], X_3D[limite:]
y_train_lstm, y_test_lstm = y_raw[:limite], y_raw[limite:]

# =====================================================================
# 3. CONSTRUCCIÓN DE LA ARQUITECTURA DE LA RED NEURONAL LSTM
# =====================================================================
tf.random.set_seed(42) # Fijar semilla para que el resultado sea replicable

modelo_lstm = Sequential([
    # Capa LSTM con 32 neuronas recurrentes
    LSTM(32, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2]), return_sequences=False),
    # Capa de regularización para evitar sobreajuste (overfitting)
    Dropout(0.1),
    # Capa densa de salida (predice un único valor continuo de glucosa)
    Dense(1)
])

# Compilar con optimizador Adam y pérdida de error absoluto medio
modelo_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mae')

print("\n Entrenando la Red Neuronal LSTM profunda...")
# Entrenamos por 40 épocas para que aprenda los patrones metabólicos sin demorar demasiado
historial = modelo_lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=40,
    batch_size=4,
    validation_split=0.1,
    verbose=0 # Oculta el texto largo de cada época para mantener limpia la pantalla
)
print(" ¡Entrenamiento de la LSTM finalizado con éxito!")

# =====================================================================
# 4. EVALUACIÓN Y PREDICCIÓN DE DEEP LEARNING
# =====================================================================
pred_lstm = modelo_lstm.predict(X_test_lstm).flatten()
mae_lstm = mean_absolute_error(y_test_lstm, pred_lstm)

print("\n==================================================")
print("   DESEMPEÑO DE LA RED NEURONAL RECURRENTE LSTM ")
print("==================================================")
print(f"• MAE de la LSTM profunda en Paciente 005: {mae_lstm:.2f} mg/dL")
print("==================================================\n")

# 5. Graficar curvas de aprendizaje y predicciones
plt.figure(figsize=(12, 4.5))
plt.plot(y_test_lstm, label='Glucosa Real (Paciente 005)', color='black', marker='o', linewidth=2)
plt.plot(pred_lstm, label=f'Predicción LSTM (MAE: {mae_lstm:.2f})', linestyle='-.', color='darkviolet', linewidth=2)

plt.title('Capacidad de Predicción del Modelo Deep Learning (LSTM Recurrente)', fontsize=12, fontweight='bold')
plt.xlabel('Muestras de Evaluación Temporal', fontsize=10)
plt.ylabel('Glucosa en Sangre (mg/dL)', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Usaremos los valores reales y las predicciones del modelo personalizado que ya calculaste
# Si por alguna razón se limpió la memoria, creamos un mapeo directo basado en tu set de evaluación
y_real_clarke = y_test_005.values
y_pred_clarke = pred_pers

def graficar_rejilla_clarke(y_true, y_pred):
    plt.figure(figsize=(8, 8), dpi=100)

    # Límites de la gráfica
    plt.xlim(0, 400)
    plt.ylim(0, 400)

    # Puntos del modelo
    plt.scatter(y_true, y_pred, marker='o', color='dodgerblue', edgecolor='black', s=40, label='Predicciones Gemelo Digital')

    # Líneas teóricas de la Rejilla de Clarke
    plt.plot([0, 400], [0, 400], 'k:', alpha=0.5)
    plt.plot([0, 175/3], [70, 70], 'k-', alpha=0.3)
    plt.plot([70, 70], [0, 175/3], 'k-', alpha=0.3)
    plt.plot([70, 400], [56, 320], 'k-', alpha=0.3)
    plt.plot([180, 400], [70, 70], 'k-', alpha=0.3)
    plt.plot([70, 400], [84, 480], 'k-', alpha=0.3)
    plt.plot([240, 240], [180, 400], 'k-', alpha=0.3)
    plt.plot([0, 70], [180, 180], 'k-', alpha=0.3)
    plt.plot([0, 180], [70, 250], 'k-', alpha=0.3)
    plt.plot([240, 400], [70, 70], 'k-', alpha=0.3)
    plt.plot([80, 400], [0, 160], 'k-', alpha=0.3)

    # Etiquetas de las Zonas Clínicas
    plt.text(30, 35, 'A', fontsize=15, fontweight='bold', color='green')
    plt.text(30, 140, 'B', fontsize=15, fontweight='bold', color='darkorange')
    plt.text(150, 350, 'D', fontsize=15, fontweight='bold', color='red')
    plt.text(350, 260, 'B', fontsize=15, fontweight='bold', color='darkorange')
    plt.text(350, 140, 'C', fontsize=15, fontweight='bold', color='crimson')
    plt.text(350, 50, 'E', fontsize=15, fontweight='bold', color='darkred')
    plt.text(30, 300, 'E', fontsize=15, fontweight='bold', color='darkred')
    plt.text(160, 20, 'C', fontsize=15, fontweight='bold', color='crimson')
    plt.text(260, 120, 'D', fontsize=15, fontweight='bold', color='red')

    plt.title('Rejilla de Error de Clarke (Evaluación Clínica del Gemelo Digital)', fontsize=12, fontweight='bold')
    plt.xlabel('Glucosa Real de Referencia (mg/dL)', fontsize=11)
    plt.ylabel('Glucosa Predicha por el Modelo (mg/dL)', fontsize=11)
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(loc='upper left')
    plt.show()

# Ejecutar la gráfica clínica
graficar_rejilla_clarke(y_real_clarke, y_pred_clarke)